# V12 — Train YOLO11s-seg @1024 (all annotators, no flips)

Add datasets:
1. Competition MAGFiLO / Solar Filament Segmentation train data
2. Code: **`solar-filament-v12-code`**

Settings → Accelerator → **GPU**.

When finished, download **`/kaggle/working/v12_export.zip`** → PC path  
`experiments/v12/yolo11s/weights/best.pt`

In [ ]:
from pathlib import Path
print("=== /kaggle/input ===")
for p in sorted(Path("/kaggle/input").iterdir()):
    print(p.name)
    try:
        for c in sorted(p.iterdir())[:15]:
            print("  ", c.name)
    except Exception as e:
        print("  (list failed)", e)

In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# Uncomment if ultralytics missing:
# %pip install -q "ultralytics>=8.3.0,<9" PyYAML

In [ ]:
import shutil
from pathlib import Path

CODE_CANDIDATES = [
    Path("/kaggle/input/solar-filament-v12-code"),
]
for p in Path("/kaggle/input").iterdir():
    if p.is_dir() and (p / "run_v12_kaggle.py").is_file():
        CODE_CANDIDATES.insert(0, p)

CODE_IN = None
for c in CODE_CANDIDATES:
    if c.is_dir() and (c / "run_v12_kaggle.py").is_file():
        CODE_IN = c
        break
if CODE_IN is None:
    raise FileNotFoundError(
        "Could not find solar-filament-v12-code (run_v12_kaggle.py). "
        f"Inputs: {[p.name for p in Path('/kaggle/input').iterdir()]}"
    )

WORK = Path("/kaggle/working/v12_code")
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(CODE_IN, WORK)

# Dataset upload uses --dir-mode zip: expand folder zips so configs/scripts/splits/src exist
import zipfile
for zpath in sorted(WORK.glob("*.zip")):
    dest = WORK / zpath.stem
    if dest.exists():
        shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(dest)
    print("unzipped", zpath.name, "->", dest)

yaml_path = WORK / "configs" / "v12_train.yaml"
print("CODE_IN", CODE_IN)
print("WORK", WORK)
print("v12_train.yaml exists:", yaml_path.is_file())
if yaml_path.is_file():
    print(yaml_path.read_text(encoding="utf-8"))

In [ ]:
import os
import sys
from pathlib import Path

WORK = Path("/kaggle/working/v12_code")
os.chdir(WORK)
sys.path.insert(0, str(WORK))

!python run_v12_kaggle.py

In [ ]:
from pathlib import Path

zip_path = Path("/kaggle/working/v12_export.zip")
export_dir = Path("/kaggle/working/v12_export")
print("zip exists:", zip_path.is_file(), zip_path)
if export_dir.is_dir():
    for p in sorted(export_dir.iterdir()):
        print(" ", p.name, p.stat().st_size)
print("\nDownload /kaggle/working/v12_export.zip from the Output panel.")
print("On PC → experiments/v12/yolo11s/weights/best.pt")